In [ ]:

import torch
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os

# Define target paths
BASE_PATH = '/content/drive/MyDrive/defect_project/dataset/NEU Metal Surface Defects Data'
ZIP_PATH  = '/content/drive/MyDrive/defect_project/new_data.zip'

# Create the target directories in your Drive
for split in ['train', 'val']:
    for cls in ['Normal', 'Rust']:
        os.makedirs(os.path.join(BASE_PATH, split, cls), exist_ok=True)

# Unzip ONLY the normal and rust folders (ignores crack, hole, scratch)
# -j junk paths (extracts files directly into the destination folder)
!unzip -j -o "{ZIP_PATH}" "*train/normal/*" -d "{BASE_PATH}/train/Normal/"
!unzip -j -o "{ZIP_PATH}" "*val/normal/*"   -d "{BASE_PATH}/val/Normal/"
!unzip -j -o "{ZIP_PATH}" "*train/rust/*"   -d "{BASE_PATH}/train/Rust/"
!unzip -j -o "{ZIP_PATH}" "*val/rust/*"     -d "{BASE_PATH}/val/Rust/"

print("\nSuccess: Normal and Rust images extracted to Drive.")

In [ ]:
import shutil
import os

# Define the paths based on your previous cells
BASE_PATH = '/content/drive/MyDrive/defect_project/dataset/NEU Metal Surface Defects Data'
valid_folder = os.path.join(BASE_PATH, 'valid')
val_folder   = os.path.join(BASE_PATH, 'val')

print("Starting merge...")

if os.path.exists(valid_folder):
    # Loop through each defect category in 'valid'
    for class_name in os.listdir(valid_folder):
        src_class_path = os.path.join(valid_folder, class_name)
        dst_class_path = os.path.join(val_folder, class_name)

        # Ensure the destination category folder exists in 'val'
        os.makedirs(dst_class_path, exist_ok=True)

        # Move every image from 'valid/Class' to 'val/Class'
        files = os.listdir(src_class_path)
        for f in files:
            shutil.move(os.path.join(src_class_path, f), os.path.join(dst_class_path, f))

        print(f"Moved {len(files)} images from valid/{class_name} to val/{class_name}")


    os.rmdir(valid_folder)
    print("\nMerge complete! The 'valid' folder has been deleted.")
else:
    print("The 'valid' folder does not exist. It might already be merged!")

In [ ]:
import os
# Define primary directory variables
TRAIN_DIR = os.path.join(BASE_PATH, 'train')
VALID_DIR = os.path.join(BASE_PATH, 'val')
TEST_DIR  = os.path.join(BASE_PATH, 'val') # Using 'val' as placeholder for test

print("=== Final Dataset Structure ===")
for split, path in [('train', TRAIN_DIR), ('val', VALID_DIR)]:
    classes = sorted(os.listdir(path))
    total = sum(len(os.listdir(os.path.join(path, c))) for c in classes)
    print(f"\n{split}/ ({total} total images)")
    for c in classes:
        n = len(os.listdir(os.path.join(path, c)))
        print(f"   {c}: {n} images")

In [ ]:
from torchvision import transforms

train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.Grayscale(num_output_channels=3),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.Grayscale(num_output_channels=3),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

In [ ]:
from torchvision import datasets

# ImageFolder auto-assigns labels based on subfolder names
train_dataset = datasets.ImageFolder(TRAIN_DIR, transform=train_transform)
valid_dataset = datasets.ImageFolder(VALID_DIR, transform=val_transform)
test_dataset  = datasets.ImageFolder(TEST_DIR,  transform=val_transform)

print(f"Classes : {train_dataset.classes}")   # should show all 8 defect types
print(f"Train   : {len(train_dataset)} images")
print(f"Valid   : {len(valid_dataset)} images")
print(f"Test    : {len(test_dataset)} images")

In [ ]:
from torch.utils.data import DataLoader

BATCH_SIZE = 32 # Adjust based on your GPU memory; 32 is safe for ResNet18

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
valid_loader = DataLoader(valid_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False)

# Quick check
dataiter = iter(train_loader)
images, labels = next(dataiter)
print(f"Batch shape: {images.shape}") # Should be [32, 3, 224, 224]

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

def imshow(tensor):
    mean = np.array([0.485, 0.456, 0.406])
    std  = np.array([0.229, 0.224, 0.225])
    img  = tensor.numpy().transpose(1, 2, 0)  # convert from CxHxW to HxWxC for matplotlib
    img  = std * img + mean                   # undo normalization so image looks correct
    return np.clip(img, 0, 1)                 # clip pixel values to valid range

images, labels = next(iter(train_loader))  # grab one batch from the loader

fig, axes = plt.subplots(2, 6, figsize=(14, 5))
for i, ax in enumerate(axes.flat):
    ax.imshow(imshow(images[i]))
    ax.set_title(train_dataset.classes[labels[i]], fontsize=9)  # show class name as title
    ax.axis('off')
plt.tight_layout()
plt.show()

print(f"Batch shape : {images.shape}")    # should be [32, 3, 224, 224]
print(f"Labels      : {labels[:8].tolist()}")  # first 6 label numbers

In [ ]:
# Run this to see how many of EACH class are in your training set
from collections import Counter

# Get all labels from the dataset
all_labels = [label for _, label in train_dataset.samples]
label_counts = Counter(all_labels)

print("Class Index | Class Name | Image Count")
print("-" * 40)
for idx, count in label_counts.items():
    print(f"{idx:11} | {train_dataset.classes[idx]:10} | {count}")

In [ ]:
from torchvision import models
import torch.nn as nn


model = models.resnet18(weights='DEFAULT')

num_features = model.fc.in_features
model.fc = nn.Linear(num_features, 8)

model = model.to(device)

print(model.fc)

In [ ]:
import torch.optim as optim
import torch.nn as nn

# 1. Loss Function: Standard for multi-class (8 classes)
criterion = nn.CrossEntropyLoss()

# 2. Strategy: Only optimize the NEW head (model.fc)
# This prevents the pre-trained ResNet 'body' from getting distorted
# while the model is still 'surprised' by the new Normal/Rust data.
optimizer = optim.Adam(model.fc.parameters(), lr=0.001)

# 3. Learning Rate Scheduler:
# Drops the learning rate by 10x every 5 epochs to 'fine-tune' the results.
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.1)

print("Setup Complete for 8-Class Training:")
print(f"Loss Function : {criterion}")
print(f"Optimizer     : Adam (Targeting model.fc only)")
print(f"LR Scheduler  : StepLR (Gamma 0.1 every 5 epochs)")

In [ ]:
EPOCHS = 15
SAVE_PATH = '/content/drive/MyDrive/defect_project/best_model.pth'

best_val_acc = 0.0  # track best accuracy to save best model

for epoch in range(EPOCHS):

    # ── Training phase ──────────────────────────────────────
    model.train()  # set model to training mode
    train_loss = 0.0
    train_correct = 0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)  # move batch to GPU

        optimizer.zero_grad()               # clear gradients from previous step
        outputs = model(images)             # forward pass
        loss = criterion(outputs, labels)   # compute loss
        loss.backward()                     # backward pass (compute gradients)
        optimizer.step()                    # update model weights

        train_loss += loss.item()
        _, preds = torch.max(outputs, 1)        # get predicted class index
        train_correct += (preds == labels).sum().item()  # count correct predictions

    # ── Validation phase ─────────────────────────────────────
    model.eval()  # set model to evaluation mode (disables dropout, batchnorm updates)
    val_loss = 0.0
    val_correct = 0

    with torch.no_grad():  # no gradient computation during validation
        for images, labels in valid_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)

            val_loss += loss.item()
            _, preds = torch.max(outputs, 1)
            val_correct += (preds == labels).sum().item()

    # ── Compute metrics ──────────────────────────────────────
    train_acc = 100 * train_correct / len(train_dataset)
    val_acc   = 100 * val_correct   / len(valid_dataset)
    avg_train_loss = train_loss / len(train_loader)
    avg_val_loss   = val_loss   / len(valid_loader)

    scheduler.step()  # update learning rate

    # ── Save best model ──────────────────────────────────────
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), SAVE_PATH)  # save to Google Drive
        saved = "saved"
    else:
        saved = ""

    print(f"Epoch {epoch+1:02d}/{EPOCHS} | "
          f"Train Loss: {avg_train_loss:.4f} | Train Acc: {train_acc:.1f}% | "
          f"Val Loss: {avg_val_loss:.4f} | Val Acc: {val_acc:.1f}% {saved}")

print(f"\nBest validation accuracy: {best_val_acc:.1f}%")


In [ ]:
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report
import matplotlib.pyplot as plt

# 1. Load the BEST model weights saved during training
model.load_state_dict(torch.load(SAVE_PATH))
model.eval()

y_true = []
y_pred = []

# 2. Run prediction on the test/validation set
with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        outputs = model(images)
        _, preds = torch.max(outputs, 1)

        y_true.extend(labels.cpu().numpy())
        y_pred.extend(preds.cpu().numpy())

# 3. Generate Confusion Matrix
cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=train_dataset.classes,
            yticklabels=train_dataset.classes)
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix: 8-Class Metal Defect Detection')
plt.show()

# 4. Detailed Report
print("\nClassification Report:\n")
print(classification_report(y_true, y_pred, target_names=train_dataset.classes))

In [ ]:
model.load_state_dict(torch.load(SAVE_PATH, map_location=device))  # load best saved weights
model.eval()  # set to evaluation mode
print("Best model loaded")

In [ ]:
all_preds  = []
all_labels = []

with torch.no_grad():  # no gradients needed during testing
    for images, labels in test_loader:
        images = images.to(device)
        outputs = model(images)               # forward pass
        _, preds = torch.max(outputs, 1)      # get predicted class index
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.numpy())

print(f"Total test images: {len(all_labels)}")

In [ ]:
from sklearn.metrics import classification_report

class_names = train_dataset.classes

print(classification_report(all_labels, all_preds, target_names=class_names))

In [ ]:

epochs = list(range(1, 16))
train_accs = [87.9, 94.1, 93.7, 97.4, 96.8, 98.7, 99.2, 99.2, 99.2, 99.2, 99.5, 99.5, 99.5, 99.8, 99.6]
val_accs   = [97.2, 98.6, 100.0, 100.0, 93.1, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0]
train_losses = [0.3734, 0.2002, 0.2137, 0.0928, 0.0975, 0.0558, 0.0295, 0.0229, 0.0203, 0.0218, 0.0208, 0.0147, 0.0160, 0.0121, 0.0138]
val_losses   = [0.0325, 0.0410, 0.0034, 0.0034, 0.1296, 0.0011, 0.0006, 0.0004, 0.0002, 0.0002, 0.0001, 0.0002, 0.0001, 0.0001, 0.0001]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(epochs, train_accs,  label='Train accuracy')
ax1.plot(epochs, val_accs,    label='Val accuracy')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Accuracy (%)')
ax1.set_title('Accuracy over epochs')
ax1.legend()
ax1.grid(True, alpha=0.3)

ax2.plot(epochs, train_losses, label='Train loss')
ax2.plot(epochs, val_losses,   label='Val loss')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Loss')
ax2.set_title('Loss over epochs')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
!pip install streamlit -q        # install streamlit
!pip install pyngrok -q          # install ngrok for public URL

In [ ]:
app_code = '''
import streamlit as st
import torch
import torch.nn as nn
from torchvision import models, transforms
from PIL import Image

# 1. UPDATED: All 8 classes in alphabetical order
CLASS_NAMES = ["Crazing", "Inclusion", "Normal", "Patches", "Pitted", "Rolled", "Rust", "Scratches"]
SAVE_PATH   = "/content/drive/MyDrive/defect_project/best_model.pth"
DEVICE      = torch.device("cuda" if torch.cuda.is_available() else "cpu")
CONFIDENCE_THRESHOLD = 0.70  # Adjusted slightly for better real-world testing

@st.cache_resource
def load_model():
    # 2. UPDATED: weights='DEFAULT' and 8 output features
    model = models.resnet18(weights=None)
    num_ftrs = model.fc.in_features
    model.fc = nn.Linear(num_ftrs, 8)

    model.load_state_dict(torch.load(SAVE_PATH, map_location=DEVICE))
    model.to(DEVICE)
    model.eval()
    return model

# 3. UPDATED: Added Grayscale(3) to match training transforms
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.Grayscale(num_output_channels=3),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

def predict(image, model):
    img = transform(image).unsqueeze(0).to(DEVICE)
    with torch.no_grad():
        outputs = model(img)
        probs   = torch.softmax(outputs, dim=1)[0]
    # 4. UPDATED: Range changed to 8
    return {CLASS_NAMES[i]: float(probs[i]) for i in range(8)}

st.set_page_config(page_title="Metal Defect Detector", layout="wide")
st.title(" Metal Surface Defect Detector")
st.write("Upload an image of a metal surface to detect defects.")

model = load_model()

uploaded_file = st.file_uploader("Choose an image...", type=["jpg", "jpeg", "png", "bmp"])

if uploaded_file is not None:
    image = Image.open(uploaded_file).convert("RGB")

    col1, col2 = st.columns([1, 1])

    with col1:
        st.subheader(" Uploaded Image")
        st.image(image, use_container_width=True)

    with col2:
        st.subheader(" Prediction Result")
        results = predict(image, model)

        top_class = max(results, key=results.get)
        top_prob  = results[top_class]

        if top_prob < CONFIDENCE_THRESHOLD:
            st.warning(" Low confidence. This might not be a metal surface or the defect is unknown.")
        else:
            if top_class == "Normal":
                st.balloons()
                st.success(f"Surface is **SAFE (Normal)**")
            elif top_class == "Rust":
                st.error(f"Defect detected: **RUST**")
            else:
                st.error(f"Defect detected: **{top_class}**")

            st.write(f"**Confidence Score:** {top_prob*100:.2f}%")

        st.write("---")
        st.subheader(" Class Probabilities")
        # Sort results to show highest probability first
        sorted_results = sorted(results.items(), key=lambda x: x[1], reverse=True)
        for cls, prob in sorted_results:
            st.text(f"{cls}")
            st.progress(prob)
'''

with open("app.py", "w") as f:
    f.write(app_code)

print("app.py updated successfully with 8-class logic!")

In [ ]:

!pip install -q pyngrok
from pyngrok import ngrok
from google.colab import userdata

# Get the secret using the correct name from your sidebar
NGROK_TOKEN = userdata.get('NGROK_TOKEN')

# Set the token
ngrok.set_auth_token(NGROK_TOKEN)

print("✅ ngrok token set successfully!")

In [ ]:
import subprocess
import threading
import time

# Start streamlit in background
proc = subprocess.Popen(
    ["streamlit", "run", "app.py", "--server.port", "8501", "--server.headless", "true"],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE
)

time.sleep(4)

public_url = ngrok.connect(8501)
print(f"Your app is live at: {public_url}")